# Importação de bibliotecas

In [1]:
# ==============================================================================
# 1. IMPORTAÇÃO DE BIBLIOTECAS E PACOTES
# ==============================================================================
# 1. Bibliotecas de terceiros (manipulação de dados e redes)
import pandas as pd
import networkx as nx
import numpy as np

# 2. Bibliotecas nativas do Python e utilitários
import json
import os
from unidecode import unidecode

# Importação de dados e parametrização

In [2]:
# ==============================================================================
# 2. CONFIGURAÇÃO DE CAMINHOS (PATHS) E PARÂMETROS GERAIS
# ==============================================================================

# 1. Configuração de caminhos base
BASE_PATH = '../data/processed' 
OUTPUT_JS_BASE = '../visualization/js'

# 2. Arquivos de entrada (Inputs)
PATH_SUGGESTIONS = f'{BASE_PATH}/similarity_matrices/suggestions_filtered.parquet'
PATH_WORDS = f'{BASE_PATH}/word_matrices/words_unigram.parquet'
PATH_AREAS = f'{BASE_PATH}/lattes_looker_studio.csv'
PATH_PREEXISTING = f'{BASE_PATH}/pre_existing_labeled.parquet'

# 3. Parâmetros de corte (limiares de conexão visual)
# Valores abaixo destes limiares não serão desenhados no frontend (Sigma.js)

# Valor de corte definido como o melhor pelo teste estatístico realizado no notebook anterior
# (valor de mediana no modelo de unigramas)
SUGGEST_THRESHOLD = 0.2264

# Limiar de publicações conjuntas das colaborações preexistentes
# (aumentar para ver apenas colaborações fortes, ex: 3)
PREEXISTING_THRESHOLD = 0.0001

# Definição da função exportadora universal (parser)

In [3]:
# ==============================================================================
# 3. DEFINIÇÃO DE FUNÇÕES UTILITÁRIAS E DE EXPORTAÇÃO (PARSER)
# ==============================================================================

# 1. Função auxiliar de normalização
# A nossa "chave secreta" para match perfeito de dicionários
def normalize_name(name):
    if pd.isna(name): return name
    return unidecode(str(name)).strip().lower()

# 2. Função principal de conversão Pandas para Sigma.js
def export_graph_to_js(df_edges, output_subfolder, dict_degrees, 
                       dict_colors, dict_communities=None, 
                       df_words=None, threshold=0.0001):

    """
    Transforma matrizes de adjacência (Pandas) em listas de nós e arestas 
    formatadas em JavaScript (JSON-like) para leitura direta pelo Sigma.js.
    """
    
    TARGET_DIR = os.path.join(OUTPUT_JS_BASE, output_subfolder)
    os.makedirs(TARGET_DIR, exist_ok=True)
    
    nodes_list = []
    edges_list = []
    
    processed_researchers = set()
    active_researchers = set()

    print(f"--- Processando Grafo: {output_subfolder.upper()} ---")

    # --------------- 1. Construção das arestas ---------------
    cols = df_edges.columns
    indices = df_edges.index
    matrix_values = df_edges.to_numpy()
    
    rows, cols_idxs = np.where(matrix_values >= threshold)
    
    for r_idx, c_idx in zip(rows, cols_idxs):
        if r_idx == c_idx: continue
        
        p1, p2 = indices[r_idx], cols[c_idx]
        
        # O nome de exibição e das imagens mantêm as letras maiúsculas originais
        p1_clean = unidecode(str(p1))
        p2_clean = unidecode(str(p2))

        pair_key = tuple(sorted((p1_clean, p2_clean)))
        if pair_key in processed_researchers: continue
        
        weight = matrix_values[r_idx, c_idx]
        
        label_text = ""
        if df_words is not None:
            try:
                label_text = str(df_words.loc[p1, p2])
            except KeyError: pass
        
        edges_list.append({
            "source": p1_clean,
            "target": p2_clean,
            "options": {
                "type": "line",
                "weight": float(weight),
                "label": label_text,
                "size": 2 
            }
        })
        
        processed_researchers.add(pair_key)
        active_researchers.add(p1)
        active_researchers.add(p2)

    # --------------- 2. Construção dos nós ---------------
    for researcher in active_researchers:
        # Nome formatado para exibição e arquivo JPG
        researcher_clean = unidecode(str(researcher))

        # Nome formatado exclusivamente para o arquivo de imagem (minúsculo e com underline)
        image_filename = researcher_clean.lower().replace(" ", "_")
        
        # Chave minúscula para buscar nos dicionários sem erros
        researcher_lookup = normalize_name(researcher)

        # Busca de cor e grau usando a chave normalizada
        node_color = dict_colors.get(researcher_lookup, '#cccccc')
        degree = dict_degrees.get(researcher, 1)

        size = 8 * (degree ** 0.5)

        node_options = {
            "size": size,
            "label": researcher_clean,
            "type": "image",
            "image": f"images/{image_filename}.jpg",
            "color": node_color,
            "labelColor": node_color
        }
        
        if dict_communities is not None:
            community = dict_communities.get(researcher_lookup)
            if isinstance(community, str) and "Comunidade" in community:
                node_options["comunidade"] = int(community.split()[-1])
            elif pd.notna(community) and isinstance(community, (int, float)):
                node_options["comunidade"] = int(community)
            else:
                node_options["comunidade"] = 0

        nodes_list.append({
            "name": researcher_clean,
            "options": node_options
        })

    # --------------- 3. Escrita dos arquivos JavaScript ---------------
    with open(f'{TARGET_DIR}/researchesNodes.js', 'w', encoding='utf-8') as f:
        f.write("const nodeList = " + json.dumps(nodes_list, indent=4, ensure_ascii=False) + ";\nexport { nodeList };")
        
    with open(f'{TARGET_DIR}/researchesEdges.js', 'w', encoding='utf-8') as f:
        f.write("const edgeList = " + json.dumps(edges_list, indent=4, ensure_ascii=False) + ";\nexport { edgeList };")

    print(f"✅ Arquivos JS gerados em: {TARGET_DIR}")
    print(f"   Nodes (Nós): {len(nodes_list)} | Edges (Arestas): {len(edges_list)}\n")

# Carregamento de Metadados

In [4]:
# ==============================================================================
# 4. CARREGAMENTO E MAPEAMENTO DE METADADOS VISUAIS (CORES E COMUNIDADES)
# ==============================================================================

# 1. Mapeamento de cores por área de atuação (Currículo Lattes)
print(">>> Carregando Metadados de Área...")
try:
    df_areas = pd.read_csv(PATH_AREAS, sep=';')
    areas_dict = dict(zip(df_areas['Pesquisador'].str.strip(), df_areas['Área'].str.strip()))
    print("✅ Áreas carregadas com sucesso!\n")
except Exception as e:
    print(f"❌ Erro ao ler arquivo de áreas: {e}")
    areas_dict = {}

COLORS_BY_AREA = {
    'Biologia': '#990000',
    'Filosofia/Sociologia': '#0033ff',
    'Filosofia': '#0033ff',
    'Sociologia': '#0033ff',
    'Física': '#330000',
    'Geografia': '#330099',
    'Gestão': '#cc9999',
    'Informática': '#ffccff',
    'Letras': '#999933',
    'Pedagogia': '#00cc99',
    'Matemática': '#339966',
    'Química': '#330033',
    'Técnico Administrativo': '#000000',
    'Educação Física': '#333333',
    'História': '#FF99CC',
    'Artes': '#000099',
    'Direito': '#FF33CC'
}

# Dicionário mestre usando a função normalize_name() para a chave!
dict_colors_master = {
    normalize_name(name): COLORS_BY_AREA.get(area, '#cccccc')
    for name, area in areas_dict.items()
}

# 2. Mapeamento de comunidades do algoritmo de Louvain
# --- Comunidades das colaborações preexistentes ---
print(">>> Carregando comunidades das colaborações preexistentes...")
hist_comm_dict = {}
for i in range(1, 8):
    filename = f'../data/processed/pre_existing_community_{i}.csv' 
    try:
        df_temp = pd.read_csv(filename)
        for name in df_temp['Pesquisador']:
            if pd.notna(name):
                hist_comm_dict[normalize_name(name)] = i
        print(f"   - {filename}: Carregado.")
    except FileNotFoundError:
        pass 

# --- Comunidades das colaborações sugeridas ---
sug_comm_dict = {}
print(">>> Carregando comunidades das colaborações sugeridas...")
for i in range(1, 20): 
    filename = f'{BASE_PATH}/processed/suggestions_community_{i}.csv'
    try:
        df_temp = pd.read_csv(filename)
        for name in df_temp['Pesquisador']:
            if pd.notna(name):
                sug_comm_dict[normalize_name(name)] = i
        print(f"   - {filename}: Carregado.")
    except FileNotFoundError:
        pass

print("✅ Comunidades (Preexistentes e Sugeridas) mapeadas com sucesso!\n")

>>> Carregando Metadados de Área...
✅ Áreas carregadas com sucesso!

>>> Carregando comunidades das colaborações preexistentes...
>>> Carregando comunidades das colaborações sugeridas...
✅ Comunidades (Preexistentes e Sugeridas) mapeadas com sucesso!



# Execução do pipeline de exportação

In [5]:
# ==============================================================================
# 5. EXECUÇÃO DO PIPELINE DE EXPORTAÇÃO (MAIN EXECUTION)
# ==============================================================================

# 1. Exportação: Grafo de Colaborações Preexistentes
print(">>> INICIANDO EXPORTAÇÃO: GRAFO DE COLABORAÇÕES PREEXISTENTES...")

df_adj = pd.read_parquet(PATH_PREEXISTING)

# Remove espaços invisíveis nos índices para evitar falhas no match de dicionários
df_adj.columns = df_adj.columns.str.strip()
df_adj.index = df_adj.columns

df_adj = df_adj.fillna(0)

# Cálculo do grau (baseando-se na frequência da colaboração preexistente)
# para dimensionamento visual dinâmico
G_preexisting = nx.from_pandas_adjacency(df_adj)
hist_deg_dict = dict(G_preexisting.degree())

export_graph_to_js(
    df_edges=df_adj,
    output_subfolder='preexistentes',
    dict_degrees=hist_deg_dict,
    dict_colors=dict_colors_master,
    dict_communities=hist_comm_dict,
    df_words=None,
    threshold=PREEXISTING_THRESHOLD
)

# 2. Exportação: Grafo de Sugestões de Colaboração
print("\n>>> INICIANDO EXPORTAÇÃO: INICIANDO GRAFO DE SUGESTÕES DE COLABORAÇÃO...")

df_suggestions = pd.read_parquet(PATH_SUGGESTIONS)
df_suggestions.index = df_suggestions.columns
df_words = pd.read_parquet(PATH_WORDS)
df_words.index = df_words.columns

# Cria uma máscara para garantir que a diagonal principal (A-A) seja rigorosamente 0,
# prevenindo auto-conexão
mask_diagonal = np.eye(len(df_suggestions), dtype=bool)
df_suggestions = df_suggestions.mask(mask_diagonal, 0)

# Cálculo do grau (baseando-se na similaridade) para dimensionamento visual dinâmico
G_suggestions = nx.from_pandas_adjacency(df_suggestions > SUGGEST_THRESHOLD)
sug_deg_dict = dict(G_suggestions.degree())

# Exportação final para JavaScript
export_graph_to_js(
    df_edges=df_suggestions,
    output_subfolder='sugestoes',
    dict_degrees=sug_deg_dict,
    dict_colors=dict_colors_master,
    dict_communities=sug_comm_dict,
    df_words=df_words,
    threshold=SUGGEST_THRESHOLD
)

print("✅ Pipeline de exportação finalizado com sucesso para todos os grafos!\n")

>>> INICIANDO EXPORTAÇÃO: GRAFO DE COLABORAÇÕES PREEXISTENTES...
--- Processando Grafo: PREEXISTENTES ---
✅ Arquivos JS gerados em: ../visualization/js\preexistentes
   Nodes (Nós): 40 | Edges (Arestas): 49


>>> INICIANDO EXPORTAÇÃO: INICIANDO GRAFO DE SUGESTÕES DE COLABORAÇÃO...
--- Processando Grafo: SUGESTOES ---
✅ Arquivos JS gerados em: ../visualization/js\sugestoes
   Nodes (Nós): 25 | Edges (Arestas): 25

✅ Pipeline de exportação finalizado com sucesso para todos os grafos!

